In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta

# ── Load raw data ──
import_export      = pd.read_csv("missing_tax_type_taxpayer_sample_dataset_imports_and_exports.csv").rename(columns={"masked_importer_pin": "PIN"})
sales_purchases    = pd.read_csv("missing_tax_type_taxpayer_sample_dataset_sales_and_purchases.csv").rename(columns={"masked_pin": "PIN"})
taxpayers_tax_types_df = pd.read_csv("missing_tax_type_taxpayer_sample_dataset_registration.csv").rename(columns={"masked_pin": "PIN"})
print("Data loaded successfully.")


Data loaded successfully.


In [ ]:
# ── Define the 12-month analysis window ──

window_end   = datetime.today().replace(day=1)          # First day of current month (exclusive upper bound)
window_start = window_end - relativedelta(months=12)    # Same day, 12 months earlier

print(f"Analysis window: {window_start.strftime('%m/%Y')} to {(window_end - relativedelta(days=1)).strftime('%m/%Y')}")


Analysis window: 03/2025 to 02/2026


In [ ]:
# ──Filter data to the 12-month window ──

# Sales & purchases
sales_purchases['month_dt'] = pd.to_datetime(sales_purchases['month']).dt.tz_localize(None)
sp_in_window = sales_purchases[
    (sales_purchases['month_dt'] >= window_start) &
    (sales_purchases['month_dt'] <  window_end)
]

# Imports & exports
import_export['month_dt'] = pd.to_datetime(import_export['period'], format='%m-%Y')
import_export = import_export[import_export['type'] == 'Import'] # Focus on imports only
ie_in_window = import_export[
    (import_export['month_dt'] >= window_start) &
    (import_export['month_dt'] <  window_end)
]

# ── Aggregate totals per taxpayer over the window ──

# Sum local purchases and collect all purchased item descriptions per PIN
purchases_agg = (
    sp_in_window
    .groupby('PIN')
    .agg(
        purchases_amount=('purchases_amount', 'sum'),
        purchased_items=('purchased_items', lambda x: list(map(str, x)))
    )
    .reset_index()
)

# Sum CIF value and collect all goods descriptions per PIN
imports_agg = (
    ie_in_window
    .groupby('PIN')
    .agg(
        cif=('cif', 'sum'),
        goods_description=('goods_description', lambda x: list(map(str, x)))
    )
    .reset_index()
)

# Add a human-readable period label to both aggregations
period_label = f"{window_start.strftime('%m/%Y')} – {(window_end - relativedelta(days=1)).strftime('%m/%Y')}"
purchases_agg['period'] = period_label
imports_agg['period']   = period_label

# ── Keep only taxpayers whose activity falls in the Turnover Tax band ─
# Turnover Tax applies to annual turnover between KES 1M and KES 25M.

MIN_THRESHOLD = 1_000_000   # KES 1 million
# MAX_THRESHOLD = 25_000_000  # KES 25 million  (upper limit for Turnover Tax)

high_volume_purchasers = purchases_agg[
    purchases_agg['purchases_amount'] >= MIN_THRESHOLD
]

high_volume_importers = imports_agg[
    imports_agg['cif'] >= MIN_THRESHOLD
]

# ──Combine purchasers and importers (union, not intersection) ──
all_eligible = high_volume_importers.merge(
    high_volume_purchasers,
    on=['PIN', 'period'],
    how='outer'
)



In [ ]:
# ── Exclude taxpayers already on Turnover Tax or PIT (resident individual) or CIT
already_registered = taxpayers_tax_types_df[
    taxpayers_tax_types_df['resident_individual'].notna() |
    taxpayers_tax_types_df['turnover_tax'].notna()        |
    taxpayers_tax_types_df['corporate_income_tax'].notna()
]

unregistered_eligible = all_eligible[
    ~all_eligible['PIN'].astype(str).isin(already_registered['PIN'].astype(str))
].copy()

# ── Clean up and compute summary columns ──

# Fill NaN where a taxpayer only appeared in one of the two datasets
unregistered_eligible['purchases_amount'] = unregistered_eligible['purchases_amount'].fillna(0)
unregistered_eligible['cif']              = unregistered_eligible['cif'].fillna(0)

# Combined both imports and purchase to get full value of expenses
unregistered_eligible['Total Purchases Value'] = (
    unregistered_eligible['purchases_amount'] + unregistered_eligible['cif']
)


unregistered_eligible = (
    unregistered_eligible
    .rename(columns={
        'purchases_amount': 'Local Purchases',
        'purchased_items':  'Local Purchased Items',
        'cif':              'Importations CIF',
        'goods_description':'Imported Items',
        'period':           'Analysis Period'
    })
    [['PIN', 'Analysis Period', 'Local Purchases', 'Importations CIF',
      'Total Purchases Value', 'Local Purchased Items', 'Imported Items']]
    .sort_values('Total Purchases Value', ascending=False)
)

unregistered_eligible['Recommended Tax'] = unregistered_eligible['Total Purchases Value'].apply(lambda x: 'Turnover Tax' if x <= 25_000_000 else 'PIT/CIT')  # Suggest VAT if above Turnover Tax threshold

# FINAL JSON SAFE CONVERSION
def convert_numpy_to_native(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.int64, np.float64)):
        return obj.item()
    return obj

unregistered_eligible = unregistered_eligible.applymap(convert_numpy_to_native)

print(f"Taxpayers qualifying for Turnover Tax (unregistered): {len(unregistered_eligible)}")
unregistered_eligible


Taxpayers qualifying for Turnover Tax (unregistered): 215


/tmp/ipykernel_145464/4147411090.py:48: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  unregistered_eligible = unregistered_eligible.applymap(convert_numpy_to_native)


,PIN,Analysis Period,Local Purchases,Importations CIF,Total Purchases Value,Local Purchased Items,Imported Items,Recommended Tax
117,84f35377e67dd4eb0ef96913e517bcfbe49012b9af73f5...,03/2025 – 02/2026,0.000000e+00,2.829714e+08,2.829714e+08,NaN,[OT; 1179; TRUCK; 4; 1CX120; JAAN1R81KR7100452...,PIT/CIT
182,cc95935f0bda2148b517ad9cce86f3a226ed5b61d59c25...,03/2025 – 02/2026,0.000000e+00,1.316364e+08,1.316364e+08,NaN,[OT; 465; STEYR; KE; 1; 50609104938; LZGJABL24...,PIT/CIT
191,d363f3840a754260dc129e0ca1c419b2a72f1ff41c3ae7...,03/2025 – 02/2026,1.124161e+08,0.000000e+00,1.124161e+08,"[Unique Consignment Reference (UCR) Number, De...",NaN,PIT/CIT
44,1e52754bcf9c1620fdbdcd41cdc62763432237ce5e7b75...,03/2025 – 02/2026,0.000000e+00,8.854007e+07,8.854007e+07,NaN,"[GASOIL FOR LOCAL USE; VL; 1; 1068643; , KEROS...",PIT/CIT
134,9753d86aae8527ebe2f393d43fa5d9df469cb3d5b9b770...,03/2025 – 02/2026,0.000000e+00,5.773381e+07,5.773381e+07,NaN,[HDPE HHM5502BN; CHEVRON PHILLIPS CHEMICAL COM...,PIT/CIT
...,...,...,...,...,...,...,...,...
111,7aa13013ca07e92ee3b021f08316e8d3fbb89bc56bd4be...,03/2025 – 02/2026,0.000000e+00,1.029418e+06,1.029418e+06,NaN,[N; TERUMO; 454PCS OF HEART LUNG PACK; H; BL L...,Turnover Tax
210,e8d61153d19d78e56b7ca4450d346982d36189bf2f69a4...,03/2025 – 02/2026,0.000000e+00,1.026171e+06,1.026171e+06,NaN,[OT; 1353; ISUZU ELF TRUCK; JP; 2; 4JJ1-3U5735...,Turnover Tax
26,09be52db9abec5cde1e5acd9c607ae6b823344cfb874f2...,03/2025 – 02/2026,0.000000e+00,1.021800e+06,1.021800e+06,NaN,"[DRY MAIZE; DRY MAIZE; BG; 250; 0;, DRY BEANS;...",Turnover Tax
122,88dfa47cda5b8a8994ef05232eb01118e6234982090a3a...,03/2025 – 02/2026,0.000000e+00,1.012607e+06,1.012607e+06,NaN,[OT; 105; USED TOYOTA HARRIER YR:2019 1980CC; ...,Turnover Tax
